# Install Dependencies

In [ ]:
!pip install -q transformers==4.45.2
!pip install -q tokenizers==0.20.1
!pip install -q sentencepiece==0.2.0
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 147.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.7 MB/s eta 0:00:00


In [ ]:
!pip install -q arabert

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.0 MB/s eta 0:00:00


# Load dataset

In [ ]:
from datasets import load_dataset

data = load_dataset(
    "MBZUAI/ArabicMMLU",
    "Math (Primary School)"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd

dev = data["dev"]
test = data["test"]

dev_df = dev.to_pandas()
test_df = test.to_pandas()

df = pd.concat([dev_df, test_df])
df.columns

Index(['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question',
       'Context', 'Answer Key', 'Option 1', 'Option 2', 'Option 3', 'Option 4',
       'Option 5', 'is_few_shot'],
      dtype='object')

In [ ]:
df.dropna(inplace = True, subset = 'Question')

In [ ]:
df.shape[0]

412

In [ ]:
test_samples = pd.read_excel('sample_math.xlsx')
test_samples.head(1)

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,9294,https://www.bassmaah.com/exams/exam-attempt/2198,Jordan,STEM,Math,Primary,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,NaN,A,36,56,63,NaN,NaN,0


In [ ]:
test_samples.shape[0]

200

In [ ]:
data_filtered = df[~df["Question"].isin(test_samples["Question"])]
data_filtered .shape[0]

212

In [ ]:
data_filtered.reset_index(drop = True, inplace = True)

# Preprocess

In [ ]:
option_columns = [
    (":A", "Option 1"),
    ("B:", "Option 2"),
    ("C:", "Option 3"),
    ("D:", "Option 4"),
    ("E:", "Option 5"),
]

choices = []
for i in range(len(data_filtered)):
    options = []
    for letter, column in option_columns:
        value = data_filtered[column].iloc[i]

        options.append(f"{letter}: {value}")
    choices.append(
        ", ".join(options)
        )

data_filtered['Options'] = choices
data_filtered.head(1)

/tmp/ipykernel_18108/3159443623.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_filtered['Options'] = choices


,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot,Options
0,9195,https://www.bassmaah.com/exams/exam-attempt/2265,Jordan,STEM,Math,Primary,اكتب العدد 54 بالصورة التحليلية,NaN,C,40+ 50,5 + 40,4+ 50,None,NaN,1,":A: 40+ 50, B:: 5 + 40, C:: 4+ 50, D:: None, E..."


In [ ]:
def build_prompt(data):
    articles = []

    for questions, choices, answer in zip(data["Question"], data["Options"], data["Answer Key"]):
        inputs = {"question": questions, "choices": choices, "answer": answer}
        articles.append(inputs)

    return articles

In [ ]:
data = build_prompt(data_filtered)

# Create a Dataframe
data = pd.DataFrame(data)

In [ ]:
data.head(1)

,question,choices,answer
0,اكتب العدد 54 بالصورة التحليلية,":A: 40+ 50, B:: 5 + 40, C:: 4+ 50, D:: None, E...",C


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)

# Configuration
CHECKPOINT = "UBC-NLP/AraT5v2-base-1024"

# Convert your pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(data)

# Split dataset into 80% train and 20% validation
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

# Preprocessing

In [ ]:
# Data Preprocessing Function
def preprocess_function(examples):
    inputs = []
    for q, c in zip(examples["question"], examples["choices"]):
        # 1. Add an explicit task instruction prefix for AraT5
        # 2. Add structural formatting tokens like [سؤال] and [خيارات]
        prompt = f"حل السؤال التالي متعدد الخيارات. سؤال: {q} خيارات: {c}"
        inputs.append(prompt)

    targets = [str(a).strip() for a in examples["answer"]]

    # Tokenize inputs and labels
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=64, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Tokenize the Split Datasets
tokenized_train = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)
tokenized_eval = eval_dataset.map(
    preprocess_function, batched=True, remove_columns=eval_dataset.column_names
)

# Data Collator for Dynamic Padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True
)

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

# Define Arguments

In [ ]:
# Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./arat5_mcq_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,  # Kept small due to large 1024 context model size
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Simulates effective batch size of 8
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # Set to False if your GPU doesn't support mixed precision
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none",  # Prevents wandb login popups
)

# Initialize Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


# Train

In [ ]:
# Start Training
trainer.train()

# Save the Final Model and Tokenizer
model.save_pretrained("./best_arat5_mcq_model")
tokenizer.save_pretrained("./best_arat5_mcq_model")


Epoch,Training Loss,Validation Loss
0,15.520500,7.239427
2,5.754100,1.033789


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


('./best_arat5_mcq_model/tokenizer_config.json',
 './best_arat5_mcq_model/special_tokens_map.json',
 './best_arat5_mcq_model/spiece.model',
 './best_arat5_mcq_model/added_tokens.json',
 './best_arat5_mcq_model/tokenizer.json')

# Evaluate

In [ ]:
choices = []
for i in range(len(test_samples)):
    options = []
    for letter, column in option_columns:
        value = test_samples[column].iloc[i]

        options.append(f"{letter}: {value}")
    choices.append(
        "\, ".join(options)
        )

test_samples['Options'] = choices
test_samples = build_prompt(test_samples)

# Create a Dataframe
test_samples = pd.DataFrame(test_samples)
test_samples.head(1)

<>:9: SyntaxWarning: invalid escape sequence '\,'
<>:9: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_18108/708227735.py:9: SyntaxWarning: invalid escape sequence '\,'
  "\, ".join(options)


,question,choices,answer
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,":A: 36\, B:: 56\, C:: 63\, D:: nan\, E:: nan",A


In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)

MODEL_PATH = "./best_arat5_mcq_model"

# Load the saved model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

test_dataset = Dataset.from_pandas(test_samples)

def preprocess_test_function(examples):
  inputs = [
      f"question: {q} choices: {c}"
      for q, c in zip(examples["question"], examples["choices"])
  ]
  targets = [str(a) for a in examples["answer"]]

  model_inputs = tokenizer(
      inputs, max_length=512, truncation=True, padding=False
  )
  labels = tokenizer(
      text_target=targets, max_length=128, truncation=True, padding=False
  )

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs


# Tokenize the test dataset
tokenized_test = test_dataset.map(
    preprocess_test_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./test_predictions",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Running batch prediction on test dataframe...")
predictions = trainer.predict(tokenized_test)

raw_predictions = predictions.predictions
raw_labels = predictions.label_ids

raw_predictions = np.where(
    raw_predictions != -100, raw_predictions, tokenizer.pad_token_id
)
raw_labels = np.where(raw_labels != -100, raw_labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(
    raw_predictions, skip_special_tokens=True
)
decoded_labels = tokenizer.batch_decode(raw_labels, skip_special_tokens=True)

test_samples["predicted_answer"] = [p.strip() for p in decoded_preds]

test_samples.head()

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Running batch prediction on test dataframe...


,question,choices,answer,predicted_answer
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,":A: 36\, B:: 56\, C:: 63\, D:: nan\, E:: nan",A,A
1,جد ناتج مايلي : 6 × 3981\n,":A: 23688\, B:: 23686\, C:: 23886\, D:: 32886\...",C,A
2,عدد من منزلتین مجموعهما = ١٠,":A: 73\, B:: 71\, C:: 83\, D:: nan\, E:: nan",A,A
3,من أدوات القياس,":A: القدم\, B:: الذراع\, C:: كلاهما\, D:: nan\...",C,A
4,أي من المسائل التالية ناتج جمعها يساوي 16,":A: 9+7\, B:: 10+7\, C:: 8+9\, D:: 9+6\, E:: nan",A,A


In [ ]:
test_samples["predicted_answer"].value_counts()

,count
predicted_answer,
A,200


In [ ]:
test_samples.to_excel('QA Math by AraT5.xlsx', index = False)
test_samples.head(2)

,question,choices,answer,predicted_answer
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,":A: 36\, B:: 56\, C:: 63\, D:: nan\, E:: nan",A,A
1,جد ناتج مايلي : 6 × 3981\n,":A: 23688\, B:: 23686\, C:: 23886\, D:: 32886\...",C,A


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_samples['answer'],
                            test_samples['predicted_answer'],
                            digits =4 ))

              precision    recall  f1-score   support

           A     0.3250    1.0000    0.4906        65
           B     0.0000    0.0000    0.0000        48
           C     0.0000    0.0000    0.0000        67
           D     0.0000    0.0000    0.0000        20

    accuracy                         0.3250       200
   macro avg     0.0813    0.2500    0.1226       200
weighted avg     0.1056    0.3250    0.1594       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
